# Limpieza y preparación de datos

Tomo los archivos crudos de `data/raw/` (`X.csv`, `Y.csv`, `Z.csv` e `historico_equipos.csv`) y los dejo listos para el análisis: mismo formato de fechas y decimales, columnas con nombres consistentes, series ordenadas y alineadas en el tiempo. No modifico ningún archivo original: todo lo que genero aquí lo guardo aparte en `data/processed/`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("raw")
PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(exist_ok=True)

## Paso 1 — Carga de cada archivo

Cada archivo crudo me llegó con un formato distinto (separador, decimal, orden de fecha), así que necesito parámetros de lectura distintos para cada uno. Por ahora solo cargo los datos y dejo las fechas como `datetime`, todavía no renombro nada.

In [ ]:
# X.csv: me llega con separador y decimal estándar, fecha YYYY-MM-DD, así que la cargo directo
df_x = pd.read_csv(RAW_DIR / "X.csv")
df_x["Date"] = pd.to_datetime(df_x["Date"], format="%Y-%m-%d")

# Y.csv: me llega con BOM, separador ';' y coma como decimal, fecha D/M/YYYY
df_y = pd.read_csv(RAW_DIR / "Y.csv", sep=";", decimal=",", encoding="utf-8-sig")
df_y["Date"] = pd.to_datetime(df_y["Date"], dayfirst=True)

# Z.csv: tiene las columnas en orden invertido (Price, Date), pero uso el mismo separador/decimal que en X
df_z = pd.read_csv(RAW_DIR / "Z.csv")
df_z["Date"] = pd.to_datetime(df_z["Date"], format="%Y-%m-%d")

# historico_equipos.csv: ya me llega en formato estándar y combinado, así que la cargo tal cual
df_hist = pd.read_csv(RAW_DIR / "historico_equipos.csv")
df_hist["Date"] = pd.to_datetime(df_hist["Date"], format="%Y-%m-%d")

df_x.head(), df_y.head(), df_z.head(), df_hist.head()

## Paso 2 — Normalización a un esquema común

Renombro las columnas a nombres consistentes en minúscula (`date`, `price_x`, `price_y`, `price_z`) y ordeno cada serie de forma ascendente por fecha (X y Y me llegaron descendentes en el archivo original).

In [ ]:
df_x = (df_x.rename(columns={"Date": "date", "Price": "price_x"})
             .sort_values("date")
             .reset_index(drop=True))

df_y = (df_y.rename(columns={"Date": "date", "Price": "price_y"})
             .sort_values("date")
             .reset_index(drop=True))

df_z = (df_z.rename(columns={"Date": "date", "Price": "price_z"})
             .sort_values("date")
             .reset_index(drop=True))

df_hist = (df_hist.rename(columns={
                "Date": "date",
                "Price_X": "price_x",
                "Price_Y": "price_y",
                "Price_Z": "price_z",
                "Price_Equipo1": "price_equipo1",
                "Price_Equipo2": "price_equipo2",
            })
            .sort_values("date")
            .reset_index(drop=True))

df_x.dtypes, df_y.dtypes, df_z.dtypes, df_hist.dtypes

## Paso 3 — Validación de calidad

Antes de combinar nada, reviso cada serie por separado: rango de fechas, fechas duplicadas y valores nulos. Esto también me sirve para confirmar que el parseo de fechas/decimales no generó errores silenciosos (por ejemplo, fechas que no se hayan podido interpretar quedarían como `NaT`).

In [ ]:
series = {"X": df_x, "Y": df_y, "Z": df_z, "historico_equipos": df_hist}

for nombre, df in series.items():
    print(f"--- {nombre} ---")
    print(f"Filas: {len(df)}")
    print(f"Rango de fechas: {df['date'].min().date()} -> {df['date'].max().date()}")
    print(f"Fechas duplicadas: {df['date'].duplicated().sum()}")
    print("Valores nulos por columna:")
    print(df.isna().sum().to_string())
    print()

También reviso si hay saltos día a día inusualmente grandes (variación porcentual extrema). Esto no elimina nada automáticamente: solo los dejo listados para decidir más adelante, con criterio, si algún salto es un error de captura o un movimiento real del mercado.

In [ ]:
UMBRAL_VARIACION = 0.15  # defino 15% en un día como umbral, lo puedo ajustar según lo que observe

for nombre, df, col in [("X", df_x, "price_x"), ("Y", df_y, "price_y"), ("Z", df_z, "price_z")]:
    var_pct = df[col].pct_change().abs()
    saltos = df.loc[var_pct > UMBRAL_VARIACION, ["date", col]].assign(variacion_pct=var_pct[var_pct > UMBRAL_VARIACION])
    print(f"--- {nombre}: {len(saltos)} saltos > {UMBRAL_VARIACION:.0%} ---")
    if len(saltos):
        print(saltos.to_string(index=False))
    print()

## Paso 4 — Alineación temporal entre X, Y y Z

Cada materia prima tiene su propio calendario (no todas cotizan exactamente los mismos días). Hago un `merge` tipo *outer* por fecha para conservar toda la historia disponible de cada una, dejando `NaN` donde una serie no tiene dato ese día — así no pierdo información, pero queda explícito dónde tengo huecos.

In [ ]:
df_materias_primas = (
    df_x.merge(df_y, on="date", how="outer")
        .merge(df_z, on="date", how="outer")
        .sort_values("date")
        .reset_index(drop=True)
)

print(f"Filas totales tras el merge: {len(df_materias_primas)}")
print(f"Rango de fechas: {df_materias_primas['date'].min().date()} -> {df_materias_primas['date'].max().date()}")
print("Nulos por columna (huecos donde una serie no tenía dato ese día):")
print(df_materias_primas.isna().sum().to_string())

## Paso 5 — Chequeo de sanidad contra `historico_equipos.csv`

`historico_equipos.csv` ya traía las tres materias primas combinadas. Comparo, en las fechas donde ambos coinciden, que mi merge produce exactamente los mismos valores que ese archivo. Si las diferencias no son ~0, algo quedó mal en mi carga o en mi merge.

In [ ]:
chequeo = df_hist[["date", "price_x", "price_y", "price_z"]].merge(
    df_materias_primas[["date", "price_x", "price_y", "price_z"]],
    on="date", how="inner", suffixes=("_hist", "_merge")
)

for col in ["price_x", "price_y", "price_z"]:
    diff = (chequeo[f"{col}_hist"] - chequeo[f"{col}_merge"]).abs()
    print(f"{col}: diferencia máxima = {diff.max():.6f} (sobre {len(chequeo)} fechas comparadas)")

## Paso 6 — Dataset final para modelar

Dejo dos versiones limpias, porque me sirven para cosas distintas:

- **`materias_primas_historico_completo.csv`**: toda la historia disponible de X, Y y Z (1988–2024), que me sirve para entender el comportamiento de cada insumo por separado.
- **`dataset_equipos_limpio.csv`**: acotado a las fechas donde también tengo precio de equipo (2010–2023), que es la tabla que voy a usar para analizar la relación materias primas → precio de equipo y para modelar.

In [ ]:
dataset_equipos_limpio = df_hist[
    ["date", "price_x", "price_y", "price_z", "price_equipo1", "price_equipo2"]
].copy()

df_materias_primas.to_csv(PROCESSED_DIR / "materias_primas_historico_completo.csv", index=False)
dataset_equipos_limpio.to_csv(PROCESSED_DIR / "dataset_equipos_limpio.csv", index=False)

print("Guardado en data/processed/:")
print(" - materias_primas_historico_completo.csv:", df_materias_primas.shape)
print(" - dataset_equipos_limpio.csv:", dataset_equipos_limpio.shape)

## Resumen de los supuestos que tomé en esta limpieza

- No eliminé ningún dato por outlier: listé los saltos grandes pero los dejo en el dataset, porque no tengo evidencia de que sean errores de captura y no movimientos reales de mercado.
- El merge entre X, Y y Z lo hice tipo *outer*: conservo toda fecha en la que al menos una serie cotizó, dejando `NaN` explícito donde me falta una serie ese día (no rellené con forward-fill en esta etapa).
- Dejé `dataset_equipos_limpio.csv` acotado al rango de `historico_equipos.csv` (2010-01-04 a 2023-08-31) porque es donde tengo los precios de los equipos, que son mi variable objetivo.
- No modifiqué ningún archivo de `data/raw/`; todo lo que generé aquí vive en `data/processed/`.